# 第9章：现代循环神经网络

**参考**：《动手学深度学习》第 9 章
**受众**：了解基础 RNN、反向传播基本概念的学习者
**学习目标**：
- 理解 GRU、LSTM 的门控机制及其解决梯度消失的原理
- 掌握深度 RNN 与双向 RNN 的结构与适用场景
- 能够使用 PyTorch 高级 API 快速搭建各类现代 RNN 模型

## 目录

1. 环境准备
2. 门控循环单元（GRU）
3. 长短期记忆网络（LSTM）
4. 深度循环神经网络
5. 双向循环神经网络
6. 模型对比与小结

---
## 1. 环境准备

In [ ]:
import torch
from torch import nn
from d2l import torch as d2l

batch_size, num_steps = 32, 35
train_iter, vocab = d2l.load_data_time_machine(batch_size, num_steps)
vocab_size  = len(vocab)
num_hiddens = 256
num_inputs  = vocab_size
device      = d2l.try_gpu()

print(f'词表大小: {vocab_size}')
print(f'训练设备: {device}')

---
## 2. 门控循环单元（GRU）

### 2.1 动机：为什么需要门控？

普通 RNN 面临梯度消失/爆炸问题，难以捕获长距离依赖。
GRU（Cho et al., 2014）通过**门控机制**解决这一问题：

| 问题场景 | 对应机制 |
|---|---|
| 早期重要信息需要长期保留 | 更新门（保持旧状态） |
| 某些词元与任务无关，应跳过 | 更新门（跳过时间步） |
| 序列出现逻辑断点，需重置状态 | 重置门 |

### 2.2 门控机制详解

**① 重置门 $\mathbf{R}_t$ 与更新门 $\mathbf{Z}_t$（均使用 sigmoid）**

$$\mathbf{R}_t = \sigma(\mathbf{X}_t \mathbf{W}_{xr} + \mathbf{H}_{t-1} \mathbf{W}_{hr} + \mathbf{b}_r)$$

$$\mathbf{Z}_t = \sigma(\mathbf{X}_t \mathbf{W}_{xz} + \mathbf{H}_{t-1} \mathbf{W}_{hz} + \mathbf{b}_z)$$

两个门均使用 sigmoid，输出值域 $(0,1)$，可理解为"开关程度"。

**② 候选隐状态 $\tilde{\mathbf{H}}_t$**

$$\tilde{\mathbf{H}}_t = \tanh\bigl(\mathbf{X}_t \mathbf{W}_{xh} + (\mathbf{R}_t \odot \mathbf{H}_{t-1}) \mathbf{W}_{hh} + \mathbf{b}_h\bigr)$$

- $\odot$ 为逐元素乘（Hadamard 积）
- $\mathbf{R}_t \to 0$：忽略历史，相当于全新开始；$\mathbf{R}_t \to 1$：退化为普通 RNN

**③ 更新隐状态（凸组合）**

$$\mathbf{H}_t = \mathbf{Z}_t \odot \mathbf{H}_{t-1} + (1 - \mathbf{Z}_t) \odot \tilde{\mathbf{H}}_t$$

- $\mathbf{Z}_t \to 1$：保留旧状态，跳过当前输入 → 适合长距离依赖
- $\mathbf{Z}_t \to 0$：完全采用候选状态 → 响应当前输入

> **总结**：重置门捕获**短期依赖**，更新门捕获**长期依赖**。

### 2.3 PyTorch 简洁实现

使用 `nn.GRU` 直接实例化 GRU 层，底层使用编译好的 CUDA 算子，速度远快于手动实现。

In [ ]:
gru_layer = nn.GRU(num_inputs, num_hiddens)
gru_model = d2l.RNNModel(gru_layer, vocab_size).to(device)

num_epochs, lr = 500, 1
d2l.train_ch8(gru_model, train_iter, vocab, lr, num_epochs, device)

**观察要点**：
- 训练结束后困惑度应接近 1，说明模型基本拟合了训练集
- 与普通 RNN 相比，GRU 在相同轮次内通常能达到更低的困惑度

---
## 3. 长短期记忆网络（LSTM）

### 3.1 设计思想

LSTM（Hochreiter & Schmidhuber, 1997）比 GRU 早约 20 年提出，设计灵感来自**计算机逻辑门**。
LSTM 引入了独立的**记忆元（memory cell）** $\mathbf{C}_t$，通过三个门精细控制信息流：

| 门 | 作用 |
|---|---|
| 输入门 $\mathbf{I}_t$ | 控制有多少新信息写入记忆元 |
| 遗忘门 $\mathbf{F}_t$ | 控制保留多少旧记忆 |
| 输出门 $\mathbf{O}_t$ | 控制从记忆元读出多少到隐状态 |

### 3.2 数学推导

**① 三个门（均使用 sigmoid）**

$$\begin{aligned}
\mathbf{I}_t &= \sigma(\mathbf{X}_t \mathbf{W}_{xi} + \mathbf{H}_{t-1} \mathbf{W}_{hi} + \mathbf{b}_i) \\
\mathbf{F}_t &= \sigma(\mathbf{X}_t \mathbf{W}_{xf} + \mathbf{H}_{t-1} \mathbf{W}_{hf} + \mathbf{b}_f) \\
\mathbf{O}_t &= \sigma(\mathbf{X}_t \mathbf{W}_{xo} + \mathbf{H}_{t-1} \mathbf{W}_{ho} + \mathbf{b}_o)
\end{aligned}$$

**② 候选记忆元（tanh，值域 $(-1,1)$）**

$$\tilde{\mathbf{C}}_t = \tanh(\mathbf{X}_t \mathbf{W}_{xc} + \mathbf{H}_{t-1} \mathbf{W}_{hc} + \mathbf{b}_c)$$

**③ 更新记忆元**

$$\mathbf{C}_t = \mathbf{F}_t \odot \mathbf{C}_{t-1} + \mathbf{I}_t \odot \tilde{\mathbf{C}}_t$$

遗忘门全为 1 且输入门全为 0 时，记忆被完整保留。梯度通过**加法路径**传播，有效缓解梯度消失。

**④ 隐状态（由输出门过滤记忆元）**

$$\mathbf{H}_t = \mathbf{O}_t \odot \tanh(\mathbf{C}_t)$$

> **注意**：记忆元 $\mathbf{C}_t$ 不直接参与输出层计算，仅通过 $\mathbf{H}_t$ 间接影响输出。

### 3.3 GRU vs LSTM 结构对比

```
GRU:  输入 + 上一隐状态 → [重置门, 更新门]          → 新隐状态
LSTM: 输入 + 上一隐状态 → [输入门, 遗忘门, 输出门]  → 新记忆元 + 新隐状态
```

- `nn.GRU`  返回 `(output, h_n)`
- `nn.LSTM` 返回 `(output, (h_n, c_n))` — 隐状态包含 h 和 c 两部分
- GRU 参数量更少，计算更快；效果通常与 LSTM 相当

### 3.4 PyTorch 简洁实现

In [ ]:
lstm_layer = nn.LSTM(num_inputs, num_hiddens)
lstm_model = d2l.RNNModel(lstm_layer, vocab_size).to(device)

num_epochs, lr = 500, 1
d2l.train_ch8(lstm_model, train_iter, vocab, lr, num_epochs, device)

---
## 4. 深度循环神经网络

### 4.1 从单层到多层

类比多层感知机，将多个 RNN 层堆叠，每层输出作为下一层输入：

$$\mathbf{H}_t^{(l)} = \phi_l\!\left(\mathbf{H}_t^{(l-1)} \mathbf{W}_{xh}^{(l)} + \mathbf{H}_{t-1}^{(l)} \mathbf{W}_{hh}^{(l)} + \mathbf{b}_h^{(l)}\right)$$

其中 $\mathbf{H}_t^{(0)} = \mathbf{X}_t$，输出层只使用最顶层的隐状态：

$$\mathbf{O}_t = \mathbf{H}_t^{(L)} \mathbf{W}_{hq} + \mathbf{b}_q$$

**信息流向**：每个隐状态同时向**时间方向**（下一时刻）和**深度方向**（上层）传递。

### 4.2 直觉理解

| 层级 | 捕获的特征 |
|---|---|
| 底层 | 局部、细粒度的短期模式（字符/词级） |
| 顶层 | 全局、粗粒度的长期依赖（语义级） |

只需指定 `num_layers` 参数即可，其余接口与单层完全相同。

In [ ]:
# 深度 RNN：2 层 LSTM
num_layers = 2
deep_lstm_layer = nn.LSTM(num_inputs, num_hiddens, num_layers)
deep_lstm_model = d2l.RNNModel(deep_lstm_layer, vocab_size).to(device)

num_epochs, lr = 500, 2
d2l.train_ch8(deep_lstm_model, train_iter, vocab, lr, num_epochs, device)

**注意**：层数增加会显著降低训练速度，需适当提高学习率；对超参数更敏感，需仔细调参。

In [ ]:
# 对比：2 层 GRU
deep_gru_layer = nn.GRU(num_inputs, num_hiddens, num_layers)
deep_gru_model = d2l.RNNModel(deep_gru_layer, vocab_size).to(device)

d2l.train_ch8(deep_gru_model, train_iter, vocab, lr, num_epochs, device)

---
## 5. 双向循环神经网络

### 5.1 动机：利用上下文信息

在填空、NER 等任务中，当前位置的**后文**对预测至关重要。例如：

> 我 ___ 饿了，我可以吃半头猪。

只看前文无法确定填什么；看到后文才能填「非常」。

### 5.2 与隐马尔可夫模型的联系

HMM 中的**前向-后向算法**是这种思想的概率版本：

- 前向递归：$\pi_{t+1}(h_{t+1}) = \sum_{h_t} \pi_t(h_t) P(x_t|h_t) P(h_{t+1}|h_t)$
- 后向递归：$\rho_{t-1}(h_{t-1}) = \sum_{h_t} P(h_t|h_{t-1}) P(x_t|h_t) \rho_t(h_t)$

双向 RNN 是这种前向-后向思想的**神经网络参数化版本**。

### 5.3 结构定义

**前向隐状态**（从左到右）：

$$\overrightarrow{\mathbf{H}}_t = \phi\!\left(\mathbf{X}_t \mathbf{W}_{xh}^{(f)} + \overrightarrow{\mathbf{H}}_{t-1} \mathbf{W}_{hh}^{(f)} + \mathbf{b}_h^{(f)}\right)$$

**后向隐状态**（从右到左）：

$$\overleftarrow{\mathbf{H}}_t = \phi\!\left(\mathbf{X}_t \mathbf{W}_{xh}^{(b)} + \overleftarrow{\mathbf{H}}_{t+1} \mathbf{W}_{hh}^{(b)} + \mathbf{b}_h^{(b)}\right)$$

**拼接后输出**（隐状态维度变为 $2h$）：

$$\mathbf{H}_t = \bigl[\overrightarrow{\mathbf{H}}_t,\ \overleftarrow{\mathbf{H}}_t\bigr] \in \mathbb{R}^{n \times 2h}, \qquad \mathbf{O}_t = \mathbf{H}_t \mathbf{W}_{hq} + \mathbf{b}_q$$

### 5.4 适用场景与局限性

| 适合双向 RNN | 不适合双向 RNN |
|---|---|
| 序列编码（BERT 思想基础） | 语言模型（预测下一个词） |
| 填空、完形填空 | 实时/在线序列预测 |
| 命名实体识别（NER） | 任何只能看到历史的任务 |
| 情感分析 | |

> **核心限制**：双向 RNN 需要看到完整序列，因此**不能用于自回归语言模型**（文本生成）。

### 5.5 错误应用示例（反面教材）

将双向 LSTM 用于语言模型是**错误用法**：训练时模型能偷看未来，
但测试时逐步生成无后向信息，预测会退化为重复字符。

In [ ]:
# 错误示例：双向 LSTM 做语言模型
# 预期结果：困惑度低，但生成文本退化为重复字符（如 travellerererer...）
bi_lstm_layer = nn.LSTM(num_inputs, num_hiddens, num_layers=2, bidirectional=True)
bi_lstm_model = d2l.RNNModel(bi_lstm_layer, vocab_size).to(device)

num_epochs, lr = 500, 1
d2l.train_ch8(bi_lstm_model, train_iter, vocab, lr, num_epochs, device)

**结果分析**：输出通常退化为 `travellerererer...`

- **训练时**：模型利用了前后文，学到了"作弊"解，而非真正的语言规律
- **测试时**：逐步生成无法提供后向信息，模型陷入混乱，只能重复已生成内容

双向 RNN 应作为**编码器**（如 BERT），而非**解码器/生成器**。

---
## 6. 模型对比与小结

### 6.1 参数量对比

In [ ]:
configs = [
    ('GRU (1层)',           nn.GRU(num_inputs,  num_hiddens)),
    ('LSTM (1层)',          nn.LSTM(num_inputs, num_hiddens)),
    ('Deep-GRU (2层)',      nn.GRU(num_inputs,  num_hiddens, num_layers=2)),
    ('Deep-LSTM (2层)',     nn.LSTM(num_inputs, num_hiddens, num_layers=2)),
    ('Bi-LSTM (1层)',       nn.LSTM(num_inputs, num_hiddens, bidirectional=True)),
    ('Deep-Bi-LSTM (2层)', nn.LSTM(num_inputs, num_hiddens, num_layers=2, bidirectional=True)),
]

print(f'{"模型":<24} {"参数量":>12}')
print('-' * 38)
for name, m in configs:
    params = sum(p.numel() for p in m.parameters())
    print(f'{name:<24} {params:>12,}')

### 6.2 知识点汇总

#### 门控循环单元（GRU）
- **重置门**：控制对历史隐状态的遗忘，捕获**短期依赖**
- **更新门**：通过凸组合控制新旧状态比例，捕获**长期依赖**
- 参数量少于 LSTM，计算更快，效果通常与 LSTM 相当

#### 长短期记忆网络（LSTM）
- 独立的**记忆元** $\mathbf{C}_t$ 通过加法路径传播，有效缓解梯度消失
- **三个门**：输入门（写）、遗忘门（选择保留）、输出门（读）
- 隐状态传到输出层，记忆元属于内部信息

#### 深度循环神经网络
- 多层堆叠，不同层捕获不同粒度的时序特征
- 仅需 `num_layers` 参数，其余接口不变
- 需要更仔细的超参数调优（学习率、初始化）

#### 双向循环神经网络
- 前向 + 后向隐状态拼接，输出维度变为 $2h$
- 适合**序列编码**（BERT、NER、情感分析），不适合**序列生成**

### 6.3 选型建议

```
需要生成/预测序列？
  是 → 单向（GRU 或 LSTM），视任务复杂度选择层数
  否（编码场景） → 双向 LSTM/GRU，或直接考虑 Transformer

计算资源有限？ → GRU（参数少，速度快）
任务复杂、序列长？ → 深度 LSTM 或 Transformer
```

### 6.4 练习

1. GRU 中如果只保留重置门（去掉更新门）会发生什么？只保留更新门呢？
2. LSTM 中候选记忆元用 $\tanh$，隐状态输出也用 $\tanh$——为什么都需要限制在 $(-1,1)$？
3. 实验：将单层 LSTM 替换为 3 层 GRU，比较困惑度与训练时间。
4. 双向 LSTM 适合情感分析，试将其用于 SST-2 数据集并观察效果。